# HaemoDiff-Risk full Kaggle run

Trains the EF network, frame VAE, four conditioned diffusion models, matched and severe synthetic cohorts, utility critics, the critic-only DP run, shadow models, and the four result tables. The cohort is the full post-audit EchoNet split. This notebook does not call pytest and does not use a toy subset.

EchoNet must already be attached under `/kaggle/input`. Simulated covariates are not patient measurements. Differential privacy is applied only to the evidential critic.

In [2]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('GPU required. Set the Kaggle accelerator to GPU and run this notebook again.')
print(torch.cuda.get_device_name(0))
print('free_gb', round(torch.cuda.mem_get_info()[0] / 1024**3, 2))

Tesla T4
free_gb 14.46


In [ ]:
from pathlib import Path
import base64, io, os, shutil, subprocess, sys, zipfile

def is_repo(path):
    return (path / "pyproject.toml").is_file() and (path / "src" / "haemodiff").is_dir()

def find_repo(base):
    if not base.exists():
        return None
    if is_repo(base):
        return base
    skip = {"Videos", "videos", ".git", "__pycache__", ".pytest_cache", "runs"}
    for dirpath, dirnames, filenames in os.walk(base):
        dirnames[:] = [name for name in dirnames if name not in skip]
        if "pyproject.toml" in filenames and is_repo(Path(dirpath)):
            return Path(dirpath)
    return None

# Cursor sends this cell to the remote Kaggle kernel. The package is inside the cell
# because the kernel cannot see the project folder on the local PC.
BUNDLE = '''...'''

found = None
for base in [Path("/kaggle/working/medAIHemo"), Path.cwd(), *Path.cwd().parents, Path("/kaggle/working"), Path("/kaggle/input")]:
    found = find_repo(base)
    if found is not None:
        break

if found is None:
    parent = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
    with zipfile.ZipFile(io.BytesIO(base64.b64decode(BUNDLE))) as archive:
        archive.extractall(parent)
    found = parent / "medAIHemo"
    if not is_repo(found):
        found = find_repo(parent)
if found is None or not is_repo(found):
    raise FileNotFoundError("The embedded package did not unpack to pyproject.toml and src/haemodiff.")

root = found
if Path("/kaggle/working").exists():
    destination = Path("/kaggle/working/medAIHemo")
    if found.resolve() != destination.resolve():
        if destination.exists():
            shutil.rmtree(destination)
        shutil.copytree(found, destination, ignore=shutil.ignore_patterns(".git", ".pytest_cache", "runs", "__pycache__", "*.pyc"))
        root = destination

os.chdir(root)
sys.path.insert(0, str(root / "src"))
print("repo", root)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", ".", "--no-deps"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "opacus", "pyyaml", "scipy", "--upgrade-strategy", "only-if-needed"])


In [ ]:
from haemodiff.config import find_echonet_root, load_config
from haemodiff.train.engine import load_records, prepare
cfg = load_config('configs/kaggle.yaml')
print('EchoNet', find_echonet_root(cfg))
run_dir, device, batch_size = prepare(cfg)
records, _ = load_records(cfg, run_dir)
train_dit = [row for row in records if row['split'] == 'TRAIN' and row['status'] == 'dit']
print('run', run_dir)
print('device', device, 'batch_size', batch_size)
print('dit_train', len(train_dit), 'severe', sum(float(row['ef']) < cfg['data']['severe_ef'] for row in train_dit))
if len(train_dit) == 0:
    raise RuntimeError('Indexed training split is empty after the tracing rules.')

In [ ]:
from haemodiff.train.train_ef import train as train_ef
train_ef(cfg)

In [ ]:
from haemodiff.train.train_vae import train as train_vae
train_vae(cfg)

In [ ]:
from haemodiff.train.train_dit import train as train_dit
for mode in cfg['conditionings']:
    train_dit(cfg, mode=mode)

In [ ]:
from haemodiff.train.sample import generate
generate(cfg)

In [ ]:
from haemodiff.eval.pillars import train_utility_models
train_utility_models(cfg)

In [ ]:
from haemodiff.eval.pillars import train_privacy_models
train_privacy_models(cfg)

In [ ]:
from haemodiff.eval.pillars import write_tables
write_tables(cfg)
from pathlib import Path
table_dir = Path('/kaggle/working/haemodiff-runs') / cfg['run_id'] / 'tables'
for name in ('table1_fidelity.csv', 'table2_utility.csv', 'figure3_calibration.json', 'table3_privacy.csv'):
    path = table_dir / name
    print(path, path.is_file(), path.stat().st_size if path.is_file() else 0)